# День 4 — Классификация и метрики

## Цель
Построить **классификатор** на Titanic (выжил / не выжил) и оценить качество: `classification_report`, confusion matrix.

## Подготовка данных

- Удаляем `Cabin` (много пропусков)
- `Sex` → 0/1
- Признаки: Pclass, Sex, Age, SibSp, Parch, Fare
- Цель: `Survived` (0/1)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv('../../data/titanic.csv')
df = df.drop(columns=['Cabin'])
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df.head()

## Pipeline: imputer → scaler → модель

1. **SimpleImputer** — заполняет пропуски (Age) медианой **на train**
2. **StandardScaler** — масштабирует признаки
3. **LogisticRegression** — линейный классификатор

Pipeline гарантирует: imputer и scaler учатся только на train — нет leakage.

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, solver="liblinear", random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(X_train.shape, X_test.shape)

## Метрики классификации

- **classification_report** — precision, recall, F1 по классам
- **confusion matrix** — сколько TP, FP, FN, TN

| | Pred 0 | Pred 1 |
|---|--------|--------|
| **True 0** | TN | FP |
| **True 1** | FN | TP |

In [ ]:
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## Выводы

- Классификация: предсказываем `Survived` по 6 признакам.
- Pipeline с imputer + scaler + LogisticRegression.
- Метрики считаем на **test**, не на train.
- F1 и recall важны, если классы несбалансированы.

---

# Day 4 — Classification and metrics

## Goal
Build a **classifier** on Titanic (survived / not) and evaluate: `classification_report`, confusion matrix.

## Data preparation

- Drop `Cabin` (many missing values)
- `Sex` → 0/1
- Features: Pclass, Sex, Age, SibSp, Parch, Fare
- Target: `Survived` (0/1)

## Pipeline: imputer → scaler → model

1. **SimpleImputer** — fills missing values (Age) with median **on train**
2. **StandardScaler** — scales features
3. **LogisticRegression** — linear classifier

Pipeline ensures imputer and scaler learn on train only — no leakage.

## Classification metrics

- **classification_report** — precision, recall, F1 per class
- **confusion matrix** — TP, FP, FN, TN counts

| | Pred 0 | Pred 1 |
|---|--------|--------|
| **True 0** | TN | FP |
| **True 1** | FN | TP |

## Conclusions

- Classification: predict `Survived` from 6 features.
- Pipeline with imputer + scaler + LogisticRegression.
- Metrics on **test**, not train.
- F1 and recall matter when classes are imbalanced.